<a href="https://colab.research.google.com/github/towardsai/ai-tutor-rag-system/blob/main/notebooks/Prebuilt_Store_Bakeoff.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prebuilt Vector Stores: One Corpus, Three Embedding Configurations

*(Adapted from the course's `Selecting_Embedding_Models.ipynb` bake-off — same comparison design, but the indexes are **loaded, not rebuilt**.)*

Re-embedding the `ai_tutor_knowledge` corpus costs real time and real money — and the course notebooks were doing it again and again. `scripts/build_vector_store.py` now embeds each configuration **once** and packages the persisted ChromaDB store as a zip. This notebook is the consumer side of that deal: unzip, load, verify, and compare.

Three prebuilt stores, same 788 documents, same 512/128 token chunks, same `{doc_id}-{i}` chunk ids — only the embedding configuration differs:

| store | model | dims | normalisation |
|---|---|---|---|
| A | `gemini-embedding-001` | 3072 | none needed — the API's default width arrives unit-normalised |
| B | `gemini-embedding-001` | 1536 | **L2-normalised by the build script** — non-3072 widths are truncations and are *not* unit length ([Google's docs](https://ai.google.dev/gemini-api/docs/embeddings)) |
| C | `text-embedding-3-small` | 1536 | none needed — "OpenAI embeddings are normalized to length 1" ([OpenAI's docs](https://developers.openai.com/api/docs/guides/embeddings)) |

Because all three stores carry **identical chunk ids**, one eval set grades all of them — the same trick the embedding bake-off lesson used, now applied to dimension and provider choices.

## 🧭 What You'll Learn

- Loading a **prebuilt** Chroma store from a zip and verifying it against its `manifest.json` before trusting it — counts, dimensions, metadata, vector norms
- Why truncated Matryoshka-style embeddings must be re-normalised, and what "the API already normalises" looks like when you *measure* it
- Embedding **queries** to match each store's build configuration (task type, dimension, normalisation) — the query side of the asymmetry story
- Comparing retrieval across configurations the honest way: identical chunk ids, one shared eval set, hit rate + MRR — never raw scores across models
- Reading quality next to **storage size and build cost**, which is the actual production trade-off

## 1. Setup: Environment, Keys, and Providers

Two embedding providers power the three stores, so this notebook needs **both** `GOOGLE_API_KEY` and `OPENAI_API_KEY` (queries are embedded live against each store's configuration). The chat `PROVIDER` is used only to generate the eval questions.

In [1]:
# ============================================================
# ⚙️ Setup — environment, dependencies, API keys, provider
# ============================================================
import os
import sys

IN_COLAB = "google.colab" in sys.modules

# Chat provider (dropdown in Colab; edit the value locally) — used for eval-set
# generation only. Embedding keys are always needed for query embedding.
PROVIDER = "gemini"  # @param ["gemini", "openai", "anthropic"]

CHAT_MODEL = "gemini-3.6-flash"  # @param ["gemini-3.6-flash", "gemini-3.5-flash-lite", "gpt-5.6-luna", "claude-sonnet-5"] {allow-input: true}

_KEY_FOR = {"gemini": "GOOGLE_API_KEY", "openai": "OPENAI_API_KEY", "anthropic": "ANTHROPIC_API_KEY"}
REQUIRED_KEYS = sorted({"GOOGLE_API_KEY", "OPENAI_API_KEY", _KEY_FOR[PROVIDER]})

if IN_COLAB:
    import importlib
    import site
    import subprocess

    # Shared install profile, pinned course-wide (July 2026).
    subprocess.run(
        [
            sys.executable, "-m", "pip", "install", "-q", "-U",
            "google-genai==2.3.0",
            "openai==2.46.0",
            "anthropic==0.117.0",
            "chromadb==1.5.9",
            "tiktoken==0.13.0",
        ],
        check=True,
    )
    importlib.reload(site)  # make newly installed packages importable without a runtime restart

    # In Colab: Secrets tab (🔑 icon) → Add new secret → e.g. OPENAI_API_KEY
    from google.colab import userdata

    for key in REQUIRED_KEYS:
        os.environ[key] = userdata.get(key)

if not IN_COLAB:
    get_ipython().run_line_magic("load_ext", "autoreload")
    get_ipython().run_line_magic("autoreload", "2")

    # Locally: dependencies are installed once from the repo's requirements.
    # Keys live in a .env file at the repo root.
    from dotenv import load_dotenv

    load_dotenv()
    missing = [k for k in REQUIRED_KEYS if not os.getenv(k)]
    assert not missing, f"Missing from .env: {missing}"

print(f"✅ Setup complete — {'Colab' if IN_COLAB else 'local'} | chat: {PROVIDER} | query embedding: gemini + openai")

✅ Setup complete — local | chat: gemini | query embedding: gemini + openai


## 2. The Course Helpers — `generate()`, Plus Per-Store Query Embedding

📎 *`generate()` is the standard course helper, exactly as built in "How To Use LLMs via API". The embedding side changes shape here: instead of one `embed()` bound to one configuration, `query_embedder_for(manifest)` builds a query-embedding function from **each store's manifest**, so a query is always embedded the way that store's documents were — same model, same `output_dimensionality`, same normalisation, and Gemini's `RETRIEVAL_QUERY` task type (the asymmetry pattern from the embedding lessons).*

In [2]:
# 📎 generate() — the course helper, exactly as built in "How To Use LLMs via API".
import numpy as np
from anthropic import Anthropic
from google import genai
from google.genai import types as genai_types
from openai import OpenAI

# Course-standard default models per provider (July 2026)
MODELS = {
    "gemini": "gemini-3.6-flash",
    "openai": "gpt-5.6-luna",
    "anthropic": "claude-sonnet-5",
}

# The setup-cell form selection (or any typed model ID) overrides the default:
MODELS[PROVIDER] = CHAT_MODEL

# Both embedding providers are always needed; anthropic only if chatting with it.
gemini_client = genai.Client()
openai_client = OpenAI()
if PROVIDER == "anthropic":
    anthropic_client = Anthropic()


def generate(prompt, system=None, model=None):
    """Send one prompt to the selected PROVIDER and return the reply text."""
    if PROVIDER == "gemini":
        response = gemini_client.models.generate_content(
            model=model or MODELS["gemini"],
            contents=prompt,
            config=genai_types.GenerateContentConfig(system_instruction=system),
        )
        return response.text

    if PROVIDER == "openai":
        response = openai_client.responses.create(
            model=model or MODELS["openai"],
            instructions=system,
            input=prompt,
            reasoning={"effort": "none"},
        )
        return response.output_text

    if PROVIDER == "anthropic":
        response = anthropic_client.messages.create(
            model=model or MODELS["anthropic"],
            max_tokens=4096,
            **({"system": system} if system else {}),
            messages=[{"role": "user", "content": prompt}],
        )
        return response.content[0].text

    raise ValueError(f"Unknown PROVIDER: {PROVIDER!r}")


def query_embedder_for(manifest):
    """A query-embedding function matching ONE store's build configuration.

    Reads the manifest rather than trusting this notebook's memory: model,
    provider, actual dimension, and whether the build script L2-normalised the
    stored vectors. Queries are normalised exactly when documents were, so
    cosine and dot product agree with the store's geometry.
    """
    model = manifest["embedding_model"]
    provider = manifest["provider"]
    dims = manifest["dimensions"]["actual"]
    normalize = manifest["normalization"]["applied_by_script"]

    def embed_query(text):
        text = text.replace("\n", " ")  # same input treatment as the course embed()
        if provider == "gemini":
            result = gemini_client.models.embed_content(
                model=model,
                contents=[text],
                config=genai_types.EmbedContentConfig(
                    task_type="RETRIEVAL_QUERY",       # asymmetry: queries ≠ documents
                    output_dimensionality=dims,
                ),
            )
            vector = list(result.embeddings[0].values)
        elif provider == "openai":
            # Native width — no dimensions= argument, exactly like the course embed()
            vector = openai_client.embeddings.create(model=model, input=[text]).data[0].embedding
        else:
            raise ValueError(f"Unknown embedding provider in manifest: {provider!r}")
        if normalize:
            arr = np.asarray(vector, dtype=np.float64)
            vector = (arr / np.linalg.norm(arr)).tolist()
        return vector

    return embed_query


def search_in(store, query, top_k=5, where=None):
    """The course search() shape, pointed at one prebuilt store."""
    result = store["collection"].query(
        query_embeddings=[store["embed_query"](query)],
        n_results=top_k,
        where=where,
    )
    return [
        {"id": cid, "text": doc, "score": 1 - dist, **meta}
        for cid, doc, dist, meta in zip(
            result["ids"][0], result["documents"][0],
            result["distances"][0], result["metadatas"][0],
        )
    ]


def show_chunks(hits, max_chars=120):
    """Print retrieved chunks with scores — look before you measure."""
    for r in hits:
        preview = r["text"][:max_chars].replace("\n", " ")
        print(f"  {r['score']:.4f} | {r['title'][:40]:40} | {preview}…")

## 3. Load the Three Stores — From Zip, in a Fresh State

The zips are produced by `scripts/build_vector_store.py` and live in `notebooks/prebuilt_stores/` (they are **not** committed to git — the `.gitignore` keeps stores and zips out; distribution is handled separately). Running in Colab? Upload the three zips into `prebuilt_stores/` first, or fetch them from wherever the course hosts them once distribution is set up.

Each store is extracted **fresh** on every run — any previous extraction is deleted first — so what gets verified below is exactly what the zip contains, never a leftover state. The extraction folder `prebuilt_stores_extracted/` is disposable (and also git-ignored).

In [3]:
import json
import shutil
import zipfile
from pathlib import Path

ZIP_DIR = Path("prebuilt_stores")
EXTRACT_DIR = Path("prebuilt_stores_extracted")

# The three configurations this notebook compares (zip names carry the dataset hash)
CONFIGS = {
    "gemini-3072": "ai_tutor_knowledge-gemini-embedding-001-3072d",
    "gemini-1536-norm": "ai_tutor_knowledge-gemini-embedding-001-1536d-norm",
    "openai-1536": "ai_tutor_knowledge-text-embedding-3-small-1536d",
}


def find_zip(slug):
    """The zip for one configuration, e.g. ai_tutor_knowledge-…-3072d-0e801a3f.zip."""
    matches = sorted(p for p in ZIP_DIR.glob(f"{slug}-*.zip") if "-limit" not in p.name)
    assert matches, (
        f"No zip found for {slug!r} in {ZIP_DIR}/ — build it first:\n"
        f"  python scripts/build_vector_store.py … --output notebooks/prebuilt_stores"
    )
    return matches[-1]


def load_store(slug):
    """Unzip into a FRESH folder, read the manifest, open the collection."""
    zip_path = find_zip(slug)
    target = EXTRACT_DIR / slug
    if target.exists():
        shutil.rmtree(target)  # fresh state: verify the zip's contents, not leftovers
    EXTRACT_DIR.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(EXTRACT_DIR)
    manifest = json.loads((target / "manifest.json").read_text())

    import chromadb

    client = chromadb.PersistentClient(path=str(target / "chroma"))
    collection = client.get_collection(manifest["collection_name"])
    return {
        "slug": slug,
        "zip_path": zip_path,
        "zip_mb": zip_path.stat().st_size / 1e6,
        "dir": target,
        "manifest": manifest,
        "collection": collection,
    }


stores = {name: load_store(slug) for name, slug in CONFIGS.items()}
for name, store in stores.items():
    store["embed_query"] = query_embedder_for(store["manifest"])
    m = store["manifest"]
    print(f"{name:18} {m['embedding_model']:24} {m['dimensions']['actual']:>4}d "
          f"| {store['collection'].count():>5} vectors | zip {store['zip_mb']:.1f} MB")

gemini-3072        gemini-embedding-001     3072d |  7428 vectors | zip 147.0 MB
gemini-1536-norm   gemini-embedding-001     1536d |  7428 vectors | zip 102.8 MB
openai-1536        text-embedding-3-small   1536d |  7428 vectors | zip 97.4 MB


## 4. Trust, but Verify: Each Store Against Its Manifest

A prebuilt index is a claim, and the manifest is the claim written down. Before comparing anything, each store must prove: the **counts** match the manifest, vectors have the **dimension** the manifest promises, **metadata** survived the round trip, chunk ids follow the course `{doc_id}-{i}` scheme, and vector **norms** are unit length — by the build script's normalisation for store B, by the providers' own API behaviour for stores A and C (both documented, both *measured* here rather than assumed).

The last check is the alignment guarantee the whole comparison rests on: all three stores must contain **identical chunk ids**.

In [4]:
import numpy as np

SAMPLE = 200  # vectors sampled per store for the norm / dimension checks

all_ids = {}
for name, store in stores.items():
    m, col = store["manifest"], store["collection"]

    # 1) counts: collection vs manifest
    count = col.count()
    assert count == m["counts"]["vectors"] == m["counts"]["chunks"], (
        f"{name}: {count} vectors in the collection, manifest says {m['counts']}"
    )

    # 2) dimensions: stored vectors vs manifest
    sample = col.get(limit=min(SAMPLE, count), include=["embeddings", "metadatas"])
    widths = {len(e) for e in sample["embeddings"]}
    assert widths == {m["dimensions"]["actual"]}, f"{name}: widths {widths} ≠ manifest"

    # 3) metadata survived, under the course field names
    assert all(set(meta) == {"title", "url", "source"} and all(meta.values())
               for meta in sample["metadatas"]), f"{name}: metadata fields missing or empty"

    # 4) id scheme: '{doc_id}-{chunk_index}'
    assert all(s.count("-") >= 5 and s.rsplit("-", 1)[1].isdigit() for s in sample["ids"]), (
        f"{name}: ids do not follow the {{doc_id}}-{{i}} scheme"
    )

    # 5) norms: unit length, whatever the provenance (script vs API), measured not assumed
    norms = np.linalg.norm(np.asarray(sample["embeddings"], dtype=np.float64), axis=1)
    assert abs(norms.mean() - 1) < 2e-3 and abs(norms.min() - 1) < 2e-3 and abs(norms.max() - 1) < 2e-3, (
        f"{name}: stored vector norms not unit length (mean {norms.mean():.6f})"
    )
    provenance = ("L2-normalised by build script" if m["normalization"]["applied_by_script"]
                  else "unit-normalised by the API")
    store["norms"] = (norms.min(), norms.mean(), norms.max())

    # 6) full id set, for the cross-store alignment check below
    all_ids[name] = sorted(col.get(include=[])["ids"])

    print(f"✅ {name:18} count={count}  dims={m['dimensions']['actual']}  "
          f"norms[min/mean/max]={norms.min():.6f}/{norms.mean():.6f}/{norms.max():.6f}  ({provenance})")

names = list(stores)
assert all_ids[names[0]] == all_ids[names[1]] == all_ids[names[2]], (
    "Chunk ids differ between stores — they were not built from the same dataset + chunking!"
)
shared_ids = all_ids[names[0]]
print(f"\n✅ alignment: all three stores contain the SAME {len(shared_ids)} chunk ids")

✅ gemini-3072        count=7428  dims=3072  norms[min/mean/max]=1.000000/1.000000/1.000000  (unit-normalised by the API)
✅ gemini-1536-norm   count=7428  dims=1536  norms[min/mean/max]=1.000000/1.000000/1.000000  (L2-normalised by build script)
✅ openai-1536        count=7428  dims=1536  norms[min/mean/max]=0.999435/0.999989/1.000673  (unit-normalised by the API)

✅ alignment: all three stores contain the SAME 7428 chunk ids


**What just happened?** Every claim in the manifests held against the loaded stores — including the one that motivated this whole exercise: store B's vectors measure unit-length because the *build script* made them so (Gemini's truncated 1536-wide output is not unit length on arrival), while A and C measure unit-length because the *APIs* deliver them that way. And the three stores expose one identical id space, which is what lets a single eval set grade all of them.

## 5. Same Questions, Three Indexes, Side by Side

Before metrics, eyes. The same query goes to all three stores; each answers from its own vector space. Watch two things: *which chunks* each configuration surfaces (rankings are comparable) and *the scores* (*not* comparable across models — each has its own score distribution, as the embedding bake-off lesson showed).

In [5]:
# @title ⚙️ Comparison knobs { display-mode: "form" }
TOP_K = 5  # @param {type:"integer"}

# A shared query set spanning the corpus's seven sources (LangChain, LlamaIndex,
# OpenAI docs, Claude Code docs, LangGraph, Deep Agents, agent engineering).
QUERIES = [
    "What is Retrieval-Augmented Generation and when should I use it?",
    "How does a LangGraph checkpointer persist conversation state between turns?",
    "How do I return structured JSON output from the OpenAI Responses API?",
    "What do Claude Code hooks do and when do they run?",
    "How does LlamaIndex split documents into nodes for indexing?",
    "How can an agent decide when to call a tool instead of answering directly?",
    "What does chunk overlap do in a text splitter, and why use it?",
    "How do I filter retrieval results by metadata in a vector store?",
]

for query in QUERIES[:2]:  # eyeball two; the eval below measures all of them
    print(f"◆ {query}")
    for name, store in stores.items():
        print(f"  — {name}")
        show_chunks(search_in(store, query, top_k=3), max_chars=90)
    print()

◆ What is Retrieval-Augmented Generation and when should I use it?
  — gemini-3072
  0.7827 | Retrieval                                | --- title: Retrieval ---  Large Language Models (LLMs) are powerful, but they have two key…
  0.7806 | Lesson 9: Retrieval-Augmented Generation |  RAG system. Its primary job is to take a user's query and efficiently find the most relev…
  0.7805 | Lesson 9: Retrieval-Augmented Generation | , and clear instructions for the LLM on how to use that context to generate an accurate an…
  — gemini-1536-norm
  0.7631 | Lesson 9: Retrieval-Augmented Generation |  RAG system. Its primary job is to take a user's query and efficiently find the most relev…
  0.7628 | Retrieval                                | --- title: Retrieval ---  Large Language Models (LLMs) are powerful, but they have two key…
  0.7604 | Lesson 9: Retrieval-Augmented Generation | , and clear instructions for the LLM on how to use that context to generate an accurate an…
  — openai-1536
  0.

In [6]:
import pandas as pd


def ranking_table(query, top_k=TOP_K):
    """One row per rank, one column per store — where the configurations diverge."""
    table = {}
    for name, store in stores.items():
        hits = search_in(store, query, top_k=top_k)
        table[name] = [f"{r['id'][:8]}…-{r['id'].rsplit('-', 1)[1]} ({r['score']:.3f})" for r in hits]
    return pd.DataFrame(table, index=[f"rank {i + 1}" for i in range(top_k)])


def overlap_summary(queries, top_k=TOP_K):
    """Mean overlap of top-k id sets between each pair of stores, plus first-divergence depth."""
    pairs = [(a, b) for i, a in enumerate(stores) for b in list(stores)[i + 1:]]
    rows = []
    for a, b in pairs:
        overlaps, first_div = [], []
        for q in queries:
            ids_a = [r["id"] for r in search_in(stores[a], q, top_k=top_k)]
            ids_b = [r["id"] for r in search_in(stores[b], q, top_k=top_k)]
            overlaps.append(len(set(ids_a) & set(ids_b)) / top_k)
            div = next((i + 1 for i, (x, y) in enumerate(zip(ids_a, ids_b)) if x != y), None)
            first_div.append(div if div is not None else top_k + 1)
        rows.append({"pair": f"{a} vs {b}",
                     f"mean overlap@{top_k}": round(sum(overlaps) / len(overlaps), 3),
                     "mean first-divergence rank": round(sum(first_div) / len(first_div), 2)})
    return pd.DataFrame(rows)


display(ranking_table(QUERIES[0]))
overlap_summary(QUERIES)

,gemini-3072,gemini-1536-norm,openai-1536
rank 1,3dedf6ba…-0 (0.783),467d5a99…-3 (0.763),467d5a99…-0 (0.621)
rank 2,467d5a99…-3 (0.781),3dedf6ba…-0 (0.763),467d5a99…-14 (0.602)
rank 3,467d5a99…-4 (0.781),467d5a99…-4 (0.760),3dedf6ba…-0 (0.600)
rank 4,0d0be460…-8 (0.779),0d0be460…-8 (0.759),467d5a99…-1 (0.589)
rank 5,3dedf6ba…-3 (0.779),3dedf6ba…-3 (0.757),3dedf6ba…-9 (0.589)


,pair,mean overlap@5,mean first-divergence rank
0,gemini-3072 vs gemini-1536-norm,0.975,3.88
1,gemini-3072 vs openai-1536,0.275,1.25
2,gemini-1536-norm vs openai-1536,0.275,1.25


**What just happened?** Competent embedders mostly agree on easy queries — the interesting information is *where* they part ways. A high overlap between the two Gemini widths with divergence only deep in the ranking says truncation is cheap; a lower overlap against the OpenAI store says the model changes retrieval more than the width does. Whether any of that costs *accuracy* is what the metrics below answer — per-query rankings against gold labels, never score-gazing.

## 6. Measuring Retrieval: One Eval Set, Three Stores

**There is no labelled query set for `ai_tutor_knowledge` under the course chunking.** The repo's `rag_eval_dataset.json` and `embedding_eval_dataset.json` carry chunk ids from the *mini-llama-articles* lessons, and the legacy `rag_eval_dataset_question_context*.json` files (in the dataset repo's archive) are keyed to LlamaIndex node UUIDs from the old chunking — none of them can grade these stores.

So this notebook does exactly what the course does when no set exists (RAG-evaluation lesson, embedding bake-off lesson): **generate question–chunk pairs with an LLM** from the stored chunks, save them, and reuse the saved file on every later run. The file `prebuilt_store_eval_dataset.json` is therefore **synthetic / LLM-generated ground truth** — good for comparing retrievers against each other, not a human-labelled benchmark — and it is validated against the stores before it is trusted (`usable_eval_set()`): every gold id must exist **and its stored text must hash-match** what the question was generated from. The content hash matters — after a chunking change, most old ids still exist under the same `{doc_id}-{i}` names but hold different text, and an id-only check would let an obsolete answer key grade the new stores as a confident, meaningless score.

Because the ids are shared, the *same* pairs grade all three stores — the comparison is apples to apples by construction.

In [7]:
# @title ⚙️ Eval knobs { display-mode: "form" }
N_EVAL_QUESTIONS = 40  # @param {type:"integer"}

import hashlib
import json as _json
from pathlib import Path


def _sha(text):
    """Content fingerprint of a chunk — what makes the cached eval set chunking-proof."""
    return hashlib.sha1(text.encode()).hexdigest()[:12]

# 📎 make_qa_pairs() from "Evaluating Your RAG Pipeline", with the bake-off
#    lesson's two hardenings (normalised reply shapes, validated cache) and one
#    labelled adaptation: chunks are sampled EVENLY across the shared id space
#    instead of taking the first N, so questions cover all seven sources rather
#    than only the first documents in the file.
QA_GEN_PROMPT = """Context information is below.
---------------------
{context}
---------------------
Given ONLY the context above and no prior knowledge, write {n} quiz question(s)
that this context can answer on its own. Questions must be self-contained
(understandable without seeing the context).

Return ONLY a JSON array of question strings, e.g. ["What is X?"]"""


def _parse_json(raw):
    return _json.loads(raw.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip())


def _as_questions(parsed):
    """Coerce a model's reply into a flat list of question strings."""
    if isinstance(parsed, dict):
        parsed = next((v for v in parsed.values() if isinstance(v, list)), [])
    questions = []
    for item in parsed if isinstance(parsed, list) else []:
        if isinstance(item, dict):
            item = item.get("question") or next((v for v in item.values() if isinstance(v, str)), "")
        if isinstance(item, str) and item.strip():
            questions.append(item.strip())
    return questions


def make_qa_pairs(collection, chunk_ids, n_questions=N_EVAL_QUESTIONS):
    """Generate (question, gold chunk_id) pairs from an even sample of chunks."""
    step = max(1, len(chunk_ids) // n_questions)
    sample_ids = chunk_ids[::step][:n_questions]
    docs = collection.get(ids=sample_ids, include=["documents"])
    qa_pairs = []
    for chunk_id, text in zip(docs["ids"], docs["documents"]):
        raw = generate(QA_GEN_PROMPT.format(context=text, n=1))
        try:
            questions = _as_questions(_parse_json(raw))
        except _json.JSONDecodeError:
            continue  # skip a malformed generation rather than crash the run
        if questions:
            qa_pairs.append({"question": questions[0], "chunk_id": chunk_id, "chunk_sha": _sha(text)})
    return qa_pairs


def usable_eval_set(pairs, collection):
    """Does this cached file actually grade THIS index?

    Ids must exist AND their stored text must hash-match what each question was
    generated from. An id-only check false-passes after a chunking change:
    old and new chunks share the {doc_id}-{i} namespace, so most old ids still
    exist — pointing at different text — and every metric silently collapses
    while looking like a real result. Files without content hashes (or with
    mismatched ones) are treated as stale and regenerated."""
    if not isinstance(pairs, list) or not pairs:
        return False
    if not all(isinstance(p, dict) and {"question", "chunk_id", "chunk_sha"} <= set(p) for p in pairs):
        return False  # pre-hash or foreign-shaped files are stale by definition
    got = collection.get(ids=[p["chunk_id"] for p in pairs], include=["documents"])
    text_by_id = dict(zip(got["ids"], got["documents"]))
    ok = sum(p["chunk_id"] in text_by_id and _sha(text_by_id[p["chunk_id"]]) == p["chunk_sha"]
             for p in pairs)
    return ok >= 0.9 * len(pairs)


EVAL_PATH = Path("prebuilt_store_eval_dataset.json")  # synthetic, LLM-generated (see above)

cached = None
if EVAL_PATH.exists():
    try:
        cached = _json.loads(EVAL_PATH.read_text())
    except _json.JSONDecodeError:
        print(f"⚠️  {EVAL_PATH} is not valid JSON — regenerating")

# Valid for the comparison only if it grades EVERY store (shared ids make this one check, but check anyway)
if cached is not None and all(usable_eval_set(cached, s["collection"]) for s in stores.values()):
    qa_pairs = cached
    print(f"Loaded {len(qa_pairs)} question–chunk pairs from {EVAL_PATH}")
else:
    if cached is not None:
        print(f"⚠️  {EVAL_PATH} doesn't match these stores — regenerating")
    qa_pairs = make_qa_pairs(stores["gemini-3072"]["collection"], shared_ids)
    EVAL_PATH.write_text(_json.dumps(qa_pairs, indent=2))
    print(f"Generated and saved {len(qa_pairs)} question–chunk pairs to {EVAL_PATH} "
          f"(~{len(qa_pairs)} chat calls)")

print("each pair looks like:", qa_pairs[0])

Generated and saved 40 question–chunk pairs to prebuilt_store_eval_dataset.json (~40 chat calls)
each pair looks like: {'question': 'According to the provided text, what message is printed when the `mem_add_text` function successfully saves an episodic memory?', 'chunk_id': 'd2a1169d-3818-5507-9a58-00bf6a9fc53a-16', 'chunk_sha': '4cb145ed933b'}


In [8]:
# 📎 Hit rate and MRR from "Evaluating Your RAG Pipeline", folded into the
#    bake-off lesson's evaluator that takes ANY (query, top_k) -> hits callable.
def evaluate_retrieval(qa_pairs, search_fn, top_k=TOP_K):
    """Hit rate + MRR for one retriever over the eval set."""
    hits = 0
    rr_total = 0.0
    for pair in qa_pairs:
        ids = [r["id"] for r in search_fn(pair["question"], top_k)]
        if pair["chunk_id"] in ids:
            hits += 1
            rr_total += 1 / (ids.index(pair["chunk_id"]) + 1)
    n = len(qa_pairs)
    return {"hit_rate": hits / n, "mrr": rr_total / n, "queries": n, "top_k": top_k}


def folder_mb(path):
    return sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file()) / 1e6


rows = []
for name, store in stores.items():
    report = evaluate_retrieval(qa_pairs, lambda q, k, s=store: search_in(s, q, top_k=k))
    m = store["manifest"]
    rows.append({
        "store": name,
        "model": m["embedding_model"],
        "dims": m["dimensions"]["actual"],
        "normalised by": "script" if m["normalization"]["applied_by_script"] else "API",
        **report,
        "store MB": round(folder_mb(store["dir"]), 1),
        "zip MB": round(store["zip_mb"], 1),
        "build cost (est. $)": m["estimated_cost_usd"],
    })

pd.DataFrame(rows).set_index("store")

,model,dims,normalised by,hit_rate,mrr,queries,top_k,store MB,zip MB,build cost (est. $)
store,,,,,,,,,,
gemini-3072,gemini-embedding-001,3072,API,0.700,0.440833,40,5,229.2,147.0,0.5467
gemini-1536-norm,gemini-embedding-001,1536,script,0.700,0.442083,40,5,181.2,102.8,0.5467
openai-1536,text-embedding-3-small,1536,API,0.575,0.432083,40,5,181.2,97.4,0.0729


**Reading the results.** The table puts the decision's two sides next to each other: retrieval quality (hit rate, MRR) against what each configuration *costs* to hold — vector width drives store size, and the manifest's estimated build cost is what re-embedding would charge you again. Three honest caveats before crowning a winner:

- With ~40 synthetic questions, differences of a few points are **noise, not signal** — rerun with a larger `N_EVAL_QUESTIONS` (or fresh generations) before switching configurations on this evidence.
- The eval set is **LLM-generated**, not human-labelled: it measures "can the retriever find the chunk a question was written from", which is the standard course proxy, nothing more.
- Raw similarity **scores never compare across models**; only the per-store rankings and the metrics computed from them do.

What *would* justify each choice: the 3072-width store earns its double storage bill only if it wins clearly on your questions; the 1536 truncation is the budget Gemini option if quality holds (Matryoshka truncation usually costs little — measure, don't assume); the OpenAI store is the cheapest to build and rebuild — and both 1536-wide stores are drop-in size-compatible with the course's `EMBED_DIM = 1536` notebooks, while the 3072 store is not (a collection's width is locked at first write, as the bake-off lesson demonstrated).

## 7. The Real Ceiling: Add the Keyword Leg (BM25 + RRF)

Read the generated questions above: most are **identifier-needles** — "which environment variable", "what directory path", "under which key". An embedding is a lossy compression of meaning, and exact identifiers carry enormous meaning per character that geometry smears (the Hybrid Search lesson's point). No chunking recipe fixes that — it is a property of dense retrieval itself, which is why every build configuration plateaus in the same band on this exam.

The course's answer — and production's — is a second ranker with the *opposite* failure mode: hand-rolled **Okapi BM25** (k1=1.5, b=0.75) over a code-aware tokenizer, fused with the dense list by **Reciprocal Rank Fusion** (k=60). The three cells below lift that code verbatim from `11-Adding_Hybrid_Search.ipynb`. Because all three stores hold identical chunks, **one keyword index serves every store**, and every mode is graded on the *same frozen questions* — so this comparison has no exam noise in it at all.

In [9]:
# 📎 tokenize(), BM25Index, rrf() — from "Hybrid Search" (11-Adding_Hybrid_Search),
#    verbatim. Identical chunks in all three stores → ONE keyword index serves all;
#    only the dense leg differs per store.
import math
import re
from collections import Counter


def tokenize(text):
    """Code-aware tokenizer for BM25 — identifiers become searchable terms.

    - splits camelCase (PersistentClient → persistent client) BEFORE lowercasing
    - lowercases, then lets every non-alphanumeric character be a boundary — which
      dissolves dotted.paths, snake_case, hyphens, and punctuation in one stroke
    - KEEPS one-character terms: BM25's idf already discounts ubiquitous short words
    """
    text = re.sub(r"([a-z0-9])([A-Z])", r"\1 \2", text)
    return re.findall(r"[a-z0-9]+", text.lower())


class BM25Index:
    """Okapi BM25 over the chunks already stored in a Chroma collection.

    k1 and b are the same constants the production tutor hand-rolls in its own
    BM25Index in app/chroma_rag.py — k1=1.5 caps how much repeated terms pay,
    b=0.75 sets how hard long chunks are penalized.
    """

    def __init__(self, ids, texts, metadatas=None, k1=1.5, b=0.75):
        self.k1, self.b = k1, b
        self.ids, self.texts = ids, texts
        self.metas = metadatas or [{} for _ in ids]

        self.doc_tokens = [tokenize(t) for t in texts]         # tokenize once, at build time
        self.doc_len = [len(toks) for toks in self.doc_tokens]
        self.avg_len = sum(self.doc_len) / len(self.doc_len)
        self.tf = [Counter(toks) for toks in self.doc_tokens]  # term frequencies per chunk

        df = Counter()                                         # in how many chunks does each term appear?
        for toks in self.doc_tokens:
            df.update(set(toks))
        n = len(texts)
        # Standard Okapi idf — rare terms score high, ubiquitous terms near zero:
        self.idf = {t: math.log((n - d + 0.5) / (d + 0.5) + 1) for t, d in df.items()}

    def score_one(self, query_tokens, i):
        """BM25 score of chunk i against an already-tokenized query."""
        score = 0.0
        tf, dl = self.tf[i], self.doc_len[i]
        for term in query_tokens:
            if term not in tf:
                continue
            f = tf[term]
            score += self.idf[term] * (f * (self.k1 + 1)) / (
                f + self.k1 * (1 - self.b + self.b * dl / self.avg_len)
            )
        return score

    def search(self, query, top_k=5):
        """Score every chunk, return the top_k in the course's result shape."""
        q_tokens = tokenize(query)
        scores = [self.score_one(q_tokens, i) for i in range(len(self.ids))]
        order = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
        return [
            {"id": self.ids[i], "text": self.texts[i], "score": round(scores[i], 4), **self.metas[i]}
            for i in order[:top_k]
            if scores[i] > 0  # sharing zero query terms is not a match
        ]


def rrf(list_a, list_b, k=60):
    """Reciprocal Rank Fusion: each chunk earns 1/(k + rank) per list it appears in."""
    fused = {}
    for results in (list_a, list_b):
        for rank, r in enumerate(results, start=1):
            fused[r["id"]] = fused.get(r["id"], 0.0) + 1.0 / (k + rank)
    return sorted(fused.items(), key=lambda pair: pair[1], reverse=True)  # [(id, fused_score), …]


_first = next(iter(stores.values()))
_stored = _first["collection"].get(include=["documents", "metadatas"])
bm25 = BM25Index(_stored["ids"], _stored["documents"], _stored["metadatas"])
print(f"BM25 index: {len(_stored['ids'])} chunks | avg length {bm25.avg_len:.0f} terms "
      f"| {len(bm25.idf):,} distinct terms  (no API, built in seconds)")


def hybrid_search_in(store, query, top_k=TOP_K, fetch_k=30):
    """Dense top-fetch_k + BM25 top-fetch_k → RRF → the fused top_k, exactly."""
    dense_hits = search_in(store, query, top_k=fetch_k)
    keyword_hits = bm25.search(query, top_k=fetch_k)
    chunks = {r["id"]: r for r in keyword_hits} | {r["id"]: r for r in dense_hits}
    fused = rrf(dense_hits, keyword_hits)
    return [{**chunks[cid], "score": round(score, 5)} for cid, score in fused[:top_k]]

BM25 index: 7428 chunks | avg length 295 terms | 24,164 distinct terms  (no API, built in seconds)


In [10]:
# Same frozen questions, three retrieval modes per store — zero exam noise in
# this comparison: any gap between rows is retrieval, not question luck.
bm25_report = evaluate_retrieval(qa_pairs, lambda q, k: bm25.search(q, top_k=k))

mode_rows = []
for name, store in stores.items():
    dense = evaluate_retrieval(qa_pairs, lambda q, k, s=store: search_in(s, q, top_k=k))
    hybrid = evaluate_retrieval(qa_pairs, lambda q, k, s=store: hybrid_search_in(s, q, top_k=k))
    mode_rows.append({"store": name, "retriever": "dense", **dense})
    mode_rows.append({"store": name, "retriever": "BM25 only", **bm25_report})
    mode_rows.append({"store": name, "retriever": "hybrid (RRF)", **hybrid})

pd.DataFrame(mode_rows).set_index(["store", "retriever"])

hit_rate       mrr  queries  top_k
store            retriever                                       
gemini-3072      dense            0.700  0.440833       40      5
                 BM25 only        0.850  0.664583       40      5
                 hybrid (RRF)     0.925  0.594583       40      5
gemini-1536-norm dense            0.700  0.442083       40      5
                 BM25 only        0.850  0.664583       40      5
                 hybrid (RRF)     0.900  0.588333       40      5
openai-1536      dense            0.575  0.432083       40      5
                 BM25 only        0.850  0.664583       40      5
                 hybrid (RRF)     0.875  0.583333       40      5

**Reading this table.** The BM25-only row is identical for every store (one shared index) — it calibrates how identifier-heavy the exam is. The decisive comparison is **dense vs hybrid within each store**: same questions, same chunks, same gold labels, so the delta is pure retrieval method. If hybrid clears dense by a wide margin here — as it does in the course's own four-row ablation — the score you've been trying to reach was never a chunking or embedding-model problem; it lives in the retrieval mode, costs no API calls (BM25 is token counts and arithmetic), and is exactly what production runs (`BM25Index` + RRF in `app/chroma_rag.py`, with Cohere reranking as the next rung — the Re-Ranking lesson).

## 🔑 Key Takeaways

- **Embed once, load thereafter.** A persisted store + manifest + zip turns the corpus's embedding bill into a one-time cost per configuration; this notebook never called a document-embedding API.
- **Verify before you trust.** Counts, dimensions, metadata, and measured vector norms against the manifest — a prebuilt index that can't prove itself is a liability, not a shortcut.
- **Normalisation is configuration.** Gemini's non-3072 widths are truncations that *you* must L2-normalise (store B); Gemini's 3072 default and OpenAI's output arrive unit-length (stores A and C) — and the *query* side must always match the document side.
- **Identical chunk ids are what make a comparison honest**: same dataset, same chunking, same ids → one eval set, three verdicts. Different ids would mean measuring different corpora.
- **Compare with rankings and metrics, never raw scores**, and read quality next to size and build cost — that pair is the actual engineering decision.
- Switching the course pipeline to any of these stores is a one-line change (point `PersistentClient` at the extracted store instead of ingesting) — but only for stores whose width matches the notebook's `EMBED_DIM`.